# R2 — Aumento con CTGAN, ajustado solo sobre train

## Qué corrige

* En la v1, CTGAN se ajustó sobre **todas** las sesiones móviles originales.
  Parte de esas filas terminó en el conjunto de evaluación, así que el
  generador había visto los datos de test. Aquí se ajusta **solo con train**.
* El dataset aumentado extiende **únicamente train**. Val y test siguen siendo
  100 % originales y no se tocan.
* Se añade la prueba que la v1 nunca ejecutó: el **clasificador
  original-vs-sintético**. La v1 descartó la cópula gaussiana porque alcanzaba
  AUC = 1,00 en esa prueba, pero luego adoptó CTGAN citando un rango de
  0,55–0,70 tomado de *benchmarks* externos, sin medirlo en estos datos.

## Qué cambia con el dataset v3

* Se generan **44 columnas de comportamiento** (antes 41 útiles + 3 degeneradas).
  Ahora las tres que estaban rotas llevan información real, así que la fidelidad
  de CTGAN sobre ellas **sí se puede leer**: en la generación anterior el KS de
  `session_duration_s` daba 0,0000 y el manuscrito lo interpretó como fidelidad
  perfecta, cuando era el KS de una constante contra sí misma.
* El post-proceso ya no escribe las reglas lógicas a mano: llama a
  `vc.repair_coherence`, compartida con R3, que además respeta el **centinela
  -1** de `transaction_amount_cop` y `time_to_transaction_s`. Sin eso, CTGAN
  produce montos interpolados en sesiones sin transacción.

Requiere GPU. Instancia sugerida: `ml.g4dn.xlarge` o superior.

In [1]:
# Celda de arranque idéntica en todos los notebooks. Deja el kernel de SageMaker
# en un estado conocido: mismo directorio de trabajo, mismas semillas, mismas
# versiones. Si algo de esto cambia entre corridas, los resultados no son
# comparables aunque los notebooks sean los mismos.
import sys, os

AQUI = os.getcwd()                    # los notebooks y vishing_common.py conviven
if AQUI not in sys.path:
    sys.path.insert(0, AQUI)

import numpy as np
import pandas as pd
import vishing_common as vc

vc.set_all_seeds()                    # random, numpy y torch (+ cuDNN determinista)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)

print("dataset:", vc.RAW_FILENAME, "(esquema", vc.DATASET_VERSION + ")")
print("bucket :", vc.BUCKET, "| prefijo:", vc.PREFIX)
print("seed   :", vc.SEED, "| split:", vc.SPLIT_MODE, "| política:", vc.FEATURE_POLICY)
print("xgboost se ejecutará en:", vc.xgb_device())
vc.check_versions()

dataset: biocatch_sinthetic_data_v3.csv (esquema v3)
bucket : poc-vishing | prefijo: v2
seed   : 42 | split: grouped | política: audited
xgboost se ejecutará en: cuda
  versiones OK: numpy 2.0.2, pandas 2.2.3, scipy 1.14.1, sklearn 1.5.2, imblearn 0.12.4, xgboost 2.1.4


,paquete,instalada,esperado,ok
0,numpy,2.0.2,">=1.26,<2.1",True
1,pandas,2.2.3,">=2.1,<2.3",True
2,scipy,1.14.1,">=1.11,<1.15",True
3,sklearn,1.5.2,">=1.4,<1.6",True
4,imblearn,0.12.4,">=0.12,<0.13",True
5,xgboost,2.1.4,">=2.0,<2.2",True


In [2]:
# SageMaker reinicia el kernel entre sesiones: la instalación va en el notebook,
# con versión fijada, para que la corrida sea reproducible sin pasos manuales.
%pip install -q "sdv>=1.5,<2.0" "ctgan>=0.10"


[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
from sdv.single_table import CTGANSynthesizer
from sdv.metadata import SingleTableMetadata
import time

In [4]:
train = vc.read_parquet(vc.P.train)
cfg = vc.read_json(vc.P.feature_contract)
contract = cfg["contratos"][cfg["activo"]]
print("train:", train.shape, "| vishing: %.4f" % train[vc.TARGET].mean())

train: (60134, 60) | vishing: 0.0500


## 1. Columnas a generar

Se excluyen identificadores, metadatos y las variables de fuga. Los puntajes e
indicadores de BioCatch **no se generan en absoluto**: como nunca entran al
modelo, la v1 gastaba un paso de condicionamiento empírico por deciles que aquí
resulta innecesario.

In [5]:
excluir = set(vc.ID_COLS + vc.LEAKAGE_COLS + vc.POSTHOC_COLS + vc.REDUNDANT_V1
              + [vc.TARGET, vc.ROW_ID, vc.ORIGIN, "split"])
gen_cols = [c for c in train.columns if c not in excluir]
print("columnas a generar:", len(gen_cols))

binarias = [c for c in gen_cols
            if pd.api.types.is_numeric_dtype(train[c])
            and set(pd.unique(train[c].dropna())) <= {0, 1}]
enteras = [c for c in gen_cols
           if pd.api.types.is_integer_dtype(train[c]) and c not in binarias]
# Columnas con centinela -1: la mezcla de una masa puntual en -1 con una
# lognormal es justo lo que peor modela un GAN tabular. Se declaran aquí para
# vigilarlas en el KS y para que el post-proceso las restaure.
con_centinela = [c for c in gen_cols if c in vc.SENTINELAS]
print("binarias (categóricas para SDV):", len(binarias), "->", binarias)
print("enteras:", len(enteras))
print("con centinela -1:", con_centinela)
for c in con_centinela:
    print("   %-26s %.1f%% de las filas de train valen -1"
          % (c, 100 * (train[c] == -1).mean()))

columnas a generar: 44
binarias (categóricas para SDV): 6 -> ['is_atypical_hour', 'phone_call_active', 'remote_access_tool_detected', 'suspicious_app_detected', 'transaction_attempted', 'is_new_beneficiary']
enteras: 13
con centinela -1: ['transaction_amount_cop', 'time_to_transaction_s']
   transaction_amount_cop     38.5% de las filas de train valen -1
   time_to_transaction_s      38.5% de las filas de train valen -1


In [6]:
HP = {
    "legit":   dict(epochs=300, batch_size=500, pac=10,
                    generator_dim=(256, 256), discriminator_dim=(256, 256),
                    embedding_dim=128, generator_lr=2e-4, discriminator_lr=2e-4),
    "vishing": dict(epochs=800, batch_size=250, pac=5,
                    generator_dim=(128, 128), discriminator_dim=(128, 128),
                    embedding_dim=64,  generator_lr=1e-4, discriminator_lr=1e-4),
}

# Cuántas filas sintéticas generar por clase.
#
# Con el v3, train pasa de ~25.400 a ~60.100 sesiones, así que 500.000 filas
# sintéticas son ~8x el original en vez de ~20x. Se mantiene la cifra para que
# la comparación con la corrida anterior sea directa; si se quiere conservar el
# mismo factor de aumento habría que subirla a ~1,2 M, con el coste de GPU
# correspondiente.
N_OBJETIVO = 500_000

# Tasa de vishing objetivo en el bloque sintético. Es deliberadamente MENOR que
# la del dataset real (5,0 %): el aumento diluye, y el rebalanceo explícito se
# hace en R3, donde queda registrado como un parámetro del experimento.
TASA_OBJETIVO = 0.015
print("train original: %d filas | sintéticas a generar: %d (%.1fx)"
      % (len(train), N_OBJETIVO, N_OBJETIVO / len(train)))
for k, v in HP.items():
    print(k, v)

train original: 60134 filas | sintéticas a generar: 500000 (8.3x)
legit {'epochs': 300, 'batch_size': 500, 'pac': 10, 'generator_dim': (256, 256), 'discriminator_dim': (256, 256), 'embedding_dim': 128, 'generator_lr': 0.0002, 'discriminator_lr': 0.0002}
vishing {'epochs': 800, 'batch_size': 250, 'pac': 5, 'generator_dim': (128, 128), 'discriminator_dim': (128, 128), 'embedding_dim': 64, 'generator_lr': 0.0001, 'discriminator_lr': 0.0001}


## 2. Entrenamiento de un sintetizador por clase

In [7]:
def entrenar(clase, etiqueta):
    sub = train[train[vc.TARGET] == clase][gen_cols].copy()
    print("\n=== clase %s: %d filas de train ===" % (etiqueta, len(sub)))

    meta = SingleTableMetadata()
    meta.detect_from_dataframe(sub)
    for c in binarias:
        meta.update_column(column_name=c, sdtype="categorical")

    syn = CTGANSynthesizer(meta, cuda=True, verbose=True, **HP[etiqueta])
    t0 = time.time()
    syn.fit(sub)
    print("entrenado en %.1f min" % ((time.time() - t0) / 60))
    return syn

syn_legit   = entrenar(0, "legit")
syn_vishing = entrenar(1, "vishing")

vc.write_pickle(syn_legit,   vc.P.ctgan_legit)
vc.write_pickle(syn_vishing, vc.P.ctgan_vishing)


=== clase legit: 57125 filas de train ===


/home/ec2-user/SageMaker/vishing-remediation/vishing-venv/lib/python3.10/site-packages/sdv/single_table/base.py:183: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
/home/ec2-user/SageMaker/vishing-remediation/vishing-venv/lib/python3.10/site-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
/home/ec2-user/SageMaker/vishing-remediation/vishing-venv/lib/python3.10/site-packages/ctgan/synthesizers/_utils.py:16: FutureWarning: `cuda` parameter is deprecated and will be removed in a future release. Please use `enable_gpu` instead.
  warnings.warn(
Gen. (-03.07) | Discrim. (+00.25): 100%|██████████| 300/300 [26:35<00:00,  5.32s/it]
/home/ec2-user/SageMaker/vishing-remediation/vishing-venv/lib/python3.10/site-packages/sdv/single_table/base.py:183: FutureWarning:

entrenado en 28.9 min

=== clase vishing: 3009 filas de train ===


Gen. (-00.24) | Discrim. (-00.51): 100%|██████████| 800/800 [07:02<00:00,  1.89it/s]


entrenado en 7.3 min
  escrito s3://poc-vishing/v2/03_augmented/synthesizers/ctgan_legit.pkl
  escrito s3://poc-vishing/v2/03_augmented/synthesizers/ctgan_vishing.pkl


's3://poc-vishing/v2/03_augmented/synthesizers/ctgan_vishing.pkl'

## 3. Muestreo y post-proceso determinístico

Las restricciones de dominio se derivan de los rangos observados **en train**,
no de constantes escritas a mano.

In [8]:
n_vish = int(N_OBJETIVO * TASA_OBJETIVO)
n_legit = N_OBJETIVO - n_vish
print("a generar: %d legítimas + %d vishing" % (n_legit, n_vish))

s_legit = syn_legit.sample(num_rows=n_legit)
s_vish  = syn_vishing.sample(num_rows=n_vish)
s_legit[vc.TARGET] = 0
s_vish[vc.TARGET] = 1
sint = pd.concat([s_legit, s_vish], ignore_index=True)
print("generadas:", sint.shape)

a generar: 492500 legítimas + 7500 vishing
generadas: (500000, 45)


In [9]:
def postproceso(d, ref):
    """Dominio + coherencia lógica. `ref` = train original.

    Dos pasos separados a propósito:
      1. Dominio: recorte a los rangos observados en train, redondeo de enteros
         y umbral de las binarias. Depende de `ref`, así que vive aquí.
      2. Coherencia: `vc.repair_coherence`, compartida con R3, que aplica el
         esquema v3 (centinela -1, max >= avg, solapamiento <= duración,
         derivadas recalculadas). Es la misma función en los dos notebooks para
         que una fila de CTGAN y una de SMOTE cumplan exactamente lo mismo.
    """
    d = d.copy()

    # -- 1. dominio: rangos observados en train --
    for c in gen_cols:
        if pd.api.types.is_numeric_dtype(ref[c]):
            d[c] = d[c].clip(ref[c].min(), ref[c].max())
    for c in binarias:
        d[c] = (d[c].astype(float) >= 0.5).astype(int)
    for c in enteras:
        d[c] = np.rint(d[c].astype(float)).astype(int)

    # -- 2. coherencia del esquema v3 --
    return vc.repair_coherence(d)


antes = vc.check_coherence(sint, "sintético crudo")
sint = postproceso(sint, train)
print("post-proceso aplicado")
vc.check_coherence(sint, "sintético post-proceso")

  coherencia [sintético crudo]: 443407 violaciones -> {'overlap_sin_llamada': 114128, 'monto_sin_transaccion': 115339, 'ttt_sin_transaccion': 109862, 'max_menor_que_avg': 103648, 'dead_time_mayor_que_sesion': 430}
post-proceso aplicado
  coherencia [sintético post-proceso]: 0 violaciones


{'overlap_sin_llamada': 0,
 'monto_sin_transaccion': 0,
 'ttt_sin_transaccion': 0,
 'max_menor_que_avg': 0,
 'dead_time_mayor_que_sesion': 0,
 'ratio_fuera_de_0_1': 0}

In [10]:
# Las derivadas se recalculan dentro de repair_coherence: nunca se generan con
# el GAN, porque una derivada generada deja de ser coherente con su numerador.
# Sobre el v3 session_duration_s tiene varianza real, así que estas tres ya
# significan lo que su nombre dice.
print("session_duration_s en train: nunique=%d, sd=%.1f s"
      % (train.session_duration_s.nunique(), train.session_duration_s.std()))
print("session_duration_s sintético: nunique=%d, sd=%.1f s"
      % (sint.session_duration_s.nunique(), sint.session_duration_s.std()))
print()
print("centinelas respetados en el bloque sintético:")
for c in con_centinela:
    sin_tx = (sint.transaction_attempted == 0)
    print("   %-26s %d filas sin transacción, %d con centinela -1"
          % (c, int(sin_tx.sum()), int((sint.loc[sin_tx, c] == -1).sum())))

session_duration_s en train: nunique=29370, sd=141.7 s
session_duration_s sintético: nunique=57214, sd=135.2 s

centinelas respetados en el bloque sintético:
   transaction_amount_cop     177510 filas sin transacción, 177510 con centinela -1
   time_to_transaction_s      177510 filas sin transacción, 177510 con centinela -1


## 4. Ensamblado del train aumentado

In [11]:
sint[vc.ROW_ID] = np.arange(len(sint), dtype=np.int64) + 10_000_000  # rango propio
sint[vc.ORIGIN] = "ctgan"
sint[vc.GROUP_COL] = "SYN-" + sint[vc.ROW_ID].astype(str)
sint["split"] = "train"

comunes = [c for c in train.columns if c in sint.columns]
train_aug = pd.concat([train[comunes], sint[comunes]], ignore_index=True)

print("train aumentado:", train_aug.shape)
print(train_aug[vc.ORIGIN].value_counts().to_string())
print("tasa de vishing: %.4f" % train_aug[vc.TARGET].mean())

train aumentado: (560134, 49)
origin
ctgan       500000
original     60134
tasa de vishing: 0.0188


In [12]:
val  = vc.read_parquet(vc.P.val)
test = vc.read_parquet(vc.P.test)

vc.assert_disjoint(train_aug, val,  "train aumentado vs val")
vc.assert_disjoint(train_aug, test, "train aumentado vs test")

  OK sin fuga [train aumentado vs val]: 560,134 train / 19,673 eval, disjuntos
  OK sin fuga [train aumentado vs test]: 560,134 train / 20,193 eval, disjuntos


## 5. Validación de calidad del aumento

### 5.1 Fidelidad marginal (Kolmogorov–Smirnov)

Se compara **original de train vs sintético**, no original vs conjunto completo:
mezclar los originales dentro del lado "aumentado" sesga el KS a la baja.

In [13]:
from scipy.stats import ks_2samp

orig = train_aug[train_aug[vc.ORIGIN] == "original"]
gen  = train_aug[train_aug[vc.ORIGIN] == "ctgan"]

filas = []
for c in contract["features"]:
    if pd.api.types.is_numeric_dtype(train_aug[c]):
        st, p = ks_2samp(orig[c], gen[c])
        filas.append({"feature": c, "KS": round(float(st), 4),
                      "p": float(p), "degenerada": bool(orig[c].nunique() <= 1)})
ks = pd.DataFrame(filas).sort_values("KS", ascending=False)
display(ks.head(20))

utiles = ks[~ks.degenerada]
print("\nvariables degeneradas (KS no interpretable):", int(ks.degenerada.sum()),
      "— sobre el v3 debe ser 0")
print("KS medio: %.4f  sobre %d variables" % (utiles.KS.mean(), len(utiles)))
print("variables con KS > 0.10:", int((utiles.KS > 0.10).sum()))
print()
print("KS de las tres variables que antes estaban rotas (ahora sí se puede leer):")
for c in ["session_duration_s", "call_overlap_duration_s", "time_to_transaction_s"]:
    fila = ks[ks.feature == c]
    if len(fila):
        print("   %-26s KS = %.4f" % (c, float(fila.KS.iloc[0])))
vc.write_csv(ks, vc.P.ks_report)

,feature,KS,p,degenerada
32,hour_of_day,0.2041,0.000000e+00,False
3,keystroke_variability,0.1756,0.000000e+00,False
6,avg_touch_size_px,0.1550,0.000000e+00,False
39,transaction_amount_cop,0.1540,0.000000e+00,False
7,swipe_speed_px_s,0.1534,0.000000e+00,False
2,typing_speed_cps,0.1393,0.000000e+00,False
4,segmented_typing_ratio,0.1190,0.000000e+00,False
13,accelerometer_jerk_mean,0.1112,0.000000e+00,False
14,phone_motion_events,0.1087,0.000000e+00,False
1,avg_interkey_latency_ms,0.1080,0.000000e+00,False



variables degeneradas (KS no interpretable): 0 — sobre el v3 debe ser 0
KS medio: 0.0767  sobre 44 variables
variables con KS > 0.10: 11

KS de las tres variables que antes estaban rotas (ahora sí se puede leer):
   session_duration_s         KS = 0.0979
   call_overlap_duration_s    KS = 0.0178
   time_to_transaction_s      KS = 0.0803
  escrito s3://poc-vishing/v2/03_augmented/quality/ks_report.csv  (44 filas)


's3://poc-vishing/v2/03_augmented/quality/ks_report.csv'

### 5.2 Detectabilidad original vs sintético

**La prueba que faltaba.** Un clasificador intenta separar filas originales de
filas generadas. La lectura:

* AUC ≈ 0,5 → indistinguibles, aumento de alta calidad.
* AUC 0,55–0,70 → rango que el manuscrito atribuye a CTGAN, citando
  *benchmarks* externos.
* AUC ≈ 1,0 → trivialmente distinguibles. Es el valor que llevó a descartar la
  cópula gaussiana en la v1.

In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

n = min(len(orig), len(gen), 20000)
muestra = pd.concat([orig.sample(n, random_state=vc.SEED),
                     gen.sample(n, random_state=vc.SEED)], ignore_index=True)
Xd = muestra[contract["features"]].fillna(0)
yd = (muestra[vc.ORIGIN] == "ctgan").astype(int)

clf = RandomForestClassifier(n_estimators=200, max_depth=10,
                             random_state=vc.SEED, n_jobs=-1)
cv = StratifiedKFold(5, shuffle=True, random_state=vc.SEED)
auc = cross_val_score(clf, Xd, yd, cv=cv, scoring="roc_auc", n_jobs=-1)

print("AUC de detectabilidad: %.4f +/- %.4f  (n=%d por lado)" % (auc.mean(), auc.std(), n))
if auc.mean() > 0.90:
    veredicto = "RECHAZAR: el aumento es trivialmente distinguible"
elif auc.mean() > 0.75:
    veredicto = "DUDOSO: distinguible, declarar como limitación"
elif auc.mean() > 0.60:
    veredicto = "ACEPTABLE: dentro del rango que la literatura reporta para CTGAN"
else:
    veredicto = "BUENO: prácticamente indistinguible"
print("veredicto:", veredicto)

vc.write_json({"auc_media": float(auc.mean()), "auc_std": float(auc.std()),
               "folds": [float(a) for a in auc], "n_por_lado": int(n),
               "veredicto": veredicto,
               "ks_medio_sin_degeneradas": float(utiles.KS.mean())},
              vc.P.detectability)

AUC de detectabilidad: 0.8738 +/- 0.0034  (n=20000 por lado)
veredicto: DUDOSO: distinguible, declarar como limitación
  escrito s3://poc-vishing/v2/03_augmented/quality/detectability.json


's3://poc-vishing/v2/03_augmented/quality/detectability.json'

In [15]:
vc.write_parquet(train_aug, vc.P.train_augmented)
print()
print("R2 completo. Continuar con R3.")

  escrito s3://poc-vishing/v2/03_augmented/train_augmented.parquet  (560,134 filas x 49 cols)

R2 completo. Continuar con R3.
